In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from ensemble import (fit_nnls_weights, apply_ensemble, paired_bootstrap_delta_auc,
                       compare_ensemble_strategies, apply_fitted_strategy)
from modeling import tune_threshold, bootstrap_ci, evaluate_at_threshold

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
MODELS_ROOT = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

MODEL_NAMES = ["dt", "rf", "gb", "svm", "nb", "knn", "xgb"]

In [2]:
import json
from sklearn.metrics import roc_auc_score

def run_ensemble_comparison(dataset_tag, data_dir, models_dir):
    oof = pd.read_csv(os.path.join(data_dir, "oof_predictions_train.csv"))
    cal_pred = pd.read_csv(os.path.join(data_dir, "base_predictions_calibration.csv"))
    test_pred = pd.read_csv(os.path.join(data_dir, "base_predictions_test.csv"))

    print(f"\n{'='*50}\n{dataset_tag}\n{'='*50}")
    print("Individual OOF AUCs:")
    for name in MODEL_NAMES:
        print(f"  {name}: {roc_auc_score(oof['y_true'], oof[name]):.4f}")

    winner_name, winner_kind, winner_obj, comparison_df = compare_ensemble_strategies(
        oof, cal_pred, MODEL_NAMES, label_col="y_true", seed=SEED)

    print("\nStrategy comparison (bootstrap mean/SE, 1-SE rule applied):")
    print(comparison_df[["strategy", "calibration_auc", "bootstrap_mean_auc", "bootstrap_se", "within_1se_of_best"]])
    print(f"\nWinning strategy: {winner_name}")

    proba_cal = apply_fitted_strategy(cal_pred, MODEL_NAMES, winner_kind, winner_obj)
    thr, macro_f1 = tune_threshold(cal_pred["y_true"], proba_cal)
    print(f"Tuned threshold: {thr:.2f} (calibration macro-F1: {macro_f1:.4f})")

    proba_test = apply_fitted_strategy(test_pred, MODEL_NAMES, winner_kind, winner_obj)
    metrics = evaluate_at_threshold(test_pred["y_true"], proba_test, thr)
    auc_point, auc_lo, auc_hi = bootstrap_ci(test_pred["y_true"], proba_test,
                                              n_resamples=N_BOOTSTRAP, seed=SEED)

    print(f"\nFinal ensemble test performance ({winner_name}):")
    print(f"  Accuracy: {metrics['accuracy']:.4f}  Macro-F1: {metrics['macro_f1']:.4f}")
    print(f"  AUC-ROC: {metrics['auc_roc']:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

    best_base_name = max(MODEL_NAMES, key=lambda n: roc_auc_score(test_pred["y_true"], test_pred[n]))
    best_base_auc = roc_auc_score(test_pred["y_true"], test_pred[best_base_name])
    delta, delta_lo, delta_hi = paired_bootstrap_delta_auc(
        test_pred["y_true"], proba_test, test_pred[best_base_name], n_resamples=N_BOOTSTRAP, seed=SEED)
    print(f"\nBest individual base learner: {best_base_name} (AUC {best_base_auc:.4f})")
    print(f"Delta AUC (ensemble - best base): {delta:.4f} [{delta_lo:.4f}, {delta_hi:.4f}]")

    os.makedirs(models_dir, exist_ok=True)
    if winner_kind == "weights":
        with open(os.path.join(models_dir, "ensemble_weights.json"), "w") as f:
            json.dump({"strategy": winner_name, "weights": {k: float(v) for k, v in winner_obj.items()}}, f, indent=2)
    else:
        import joblib
        joblib.dump(winner_obj, os.path.join(models_dir, "ensemble_logistic_stacking.joblib"))
        with open(os.path.join(models_dir, "ensemble_strategy.json"), "w") as f:
            json.dump({"strategy": winner_name}, f, indent=2)

    comparison_df.to_csv(os.path.join(models_dir, "strategy_comparison.csv"), index=False)

    summary = {
        "dataset": dataset_tag, "winning_strategy": winner_name, "threshold": round(thr, 2),
        "accuracy": round(metrics["accuracy"], 4), "macro_f1": round(metrics["macro_f1"], 4),
        "auc_roc": round(metrics["auc_roc"], 4), "auc_ci_lower": round(auc_lo, 4),
        "auc_ci_upper": round(auc_hi, 4), "best_base_model": best_base_name,
        "best_base_auc": round(best_base_auc, 4), "delta_auc": round(delta, 4),
        "delta_auc_ci_lower": round(delta_lo, 4), "delta_auc_ci_upper": round(delta_hi, 4),
    }
    return summary, proba_test

In [3]:
summary_pk, proba_pk = run_ensemble_comparison(
    "pakistan", os.path.join(PROCESSED_DIR, "pakistan"), os.path.join(MODELS_ROOT, "ensemble_final", "pakistan"))


pakistan
Individual OOF AUCs:
  dt: 0.5272
  rf: 0.5382
  gb: 0.5414
  svm: 0.5418
  nb: 0.5249
  knn: 0.5291
  xgb: 0.5379

Strategy comparison (bootstrap mean/SE, 1-SE rule applied):
             strategy  calibration_auc  bootstrap_mean_auc  bootstrap_se  \
0          single_svm         0.563587            0.563045      0.010897   
1           single_rf         0.562121            0.561924      0.011020   
2                nnls         0.557924            0.557539      0.011125   
3           single_nb         0.557634            0.557164      0.011108   
4   logistic_stacking         0.555037            0.554632      0.011244   
5        equal_weight         0.552416            0.551900      0.011274   
6          single_xgb         0.549341            0.549199      0.011135   
7    greedy_selection         0.548883            0.548436      0.011182   
8       auc_optimized         0.548280            0.547792      0.011121   
9           single_dt         0.545294            0.54

In [5]:
summary_ar_dep, proba_ar_dep = run_ensemble_comparison(
    "arab5_depression", os.path.join(PROCESSED_DIR, "arab5_depression"),
    os.path.join(MODELS_ROOT, "ensemble_final", "arab5_depression"))


arab5_depression
Individual OOF AUCs:
  dt: 0.6702
  rf: 0.6868
  gb: 0.6803
  svm: 0.6742
  nb: 0.6644
  knn: 0.6792
  xgb: 0.6775

Strategy comparison (bootstrap mean/SE, 1-SE rule applied):
             strategy  calibration_auc  bootstrap_mean_auc  bootstrap_se  \
0          single_knn         0.669696            0.669209      0.029987   
1    greedy_selection         0.649258            0.648430      0.030755   
2       auc_optimized         0.647659            0.646961      0.030856   
3           single_rf         0.647842            0.646637      0.030344   
4                nnls         0.645330            0.644258      0.030578   
5   logistic_stacking         0.641356            0.640404      0.030828   
6        equal_weight         0.634871            0.633741      0.030829   
7           single_nb         0.632039            0.630766      0.031141   
8          single_xgb         0.627518            0.626498      0.031269   
9          single_svm         0.624640        

In [6]:
summary_ar_anx, proba_ar_anx = run_ensemble_comparison(
    "arab5_anxiety", os.path.join(PROCESSED_DIR, "arab5_anxiety"),
    os.path.join(MODELS_ROOT, "ensemble_final", "arab5_anxiety"))


arab5_anxiety
Individual OOF AUCs:
  dt: 0.7478
  rf: 0.7718
  gb: 0.7664
  svm: 0.7598
  nb: 0.7524
  knn: 0.7654
  xgb: 0.7644

Strategy comparison (bootstrap mean/SE, 1-SE rule applied):
             strategy  calibration_auc  bootstrap_mean_auc  bootstrap_se  \
0          single_xgb         0.740536            0.738626      0.042846   
1           single_nb         0.740272            0.738459      0.042353   
2           single_rf         0.735523            0.734075      0.043493   
3        equal_weight         0.732621            0.731169      0.043440   
4           single_gb         0.731896            0.730532      0.041473   
5                nnls         0.729851            0.728563      0.043379   
6   logistic_stacking         0.729455            0.727938      0.043167   
7    greedy_selection         0.728796            0.727482      0.043239   
8          single_svm         0.726289            0.725896      0.042747   
9       auc_optimized         0.721013           

In [7]:
summary_ug, proba_ug = run_ensemble_comparison(
    "uganda", os.path.join(PROCESSED_DIR, "uganda"), os.path.join(MODELS_ROOT, "ensemble_final", "uganda"))


uganda
Individual OOF AUCs:
  dt: 0.6553
  rf: 0.6913
  gb: 0.6899
  svm: 0.6840
  nb: 0.6432
  knn: 0.6817
  xgb: 0.6916

Strategy comparison (bootstrap mean/SE, 1-SE rule applied):
             strategy  calibration_auc  bootstrap_mean_auc  bootstrap_se  \
0           single_rf         0.686370            0.686254      0.009385   
1   logistic_stacking         0.684005            0.683832      0.009456   
2       auc_optimized         0.683717            0.683513      0.009483   
3                nnls         0.683313            0.683113      0.009492   
4    greedy_selection         0.683276            0.683102      0.009478   
5           single_gb         0.682098            0.681927      0.009478   
6          single_xgb         0.681875            0.681696      0.009409   
7        equal_weight         0.677356            0.677330      0.009602   
8          single_knn         0.667729            0.667405      0.009649   
9          single_svm         0.665764            0.6652

In [8]:
all_summaries = pd.DataFrame([summary_pk, summary_ar_dep, summary_ar_anx, summary_ug])
print(all_summaries)
all_summaries.to_csv(os.path.join(TABLES_DIR, "ensemble_summary.csv"), index=False)

            dataset winning_strategy  threshold  accuracy  macro_f1  auc_roc  \
0          pakistan       single_svm       0.46    0.5565    0.5369   0.5621   
1  arab5_depression       single_knn       0.47    0.5980    0.5974   0.6337   
2     arab5_anxiety       single_xgb       0.45    0.7853    0.7167   0.7848   
3            uganda        single_rf       0.55    0.6364    0.6314   0.6965   

   auc_ci_lower  auc_ci_upper best_base_model  best_base_auc  delta_auc  \
0        0.5391        0.5841             xgb         0.5650    -0.0029   
1        0.5680        0.6975              nb         0.6925    -0.0583   
2        0.7021        0.8626             knn         0.7976    -0.0126   
3        0.6770        0.7162              rf         0.6965     0.0000   

   delta_auc_ci_lower  delta_auc_ci_upper  
0             -0.0178              0.0119  
1             -0.1001             -0.0150  
2             -0.0575              0.0333  
3              0.0000              0.0000  


In [9]:
FINAL_MODEL = "rf"

def finalize_proposed_model(dataset_tag, data_dir):
    cal_pred = pd.read_csv(os.path.join(data_dir, "base_predictions_calibration.csv"))
    test_pred = pd.read_csv(os.path.join(data_dir, "base_predictions_test.csv"))

    proba_cal = cal_pred[FINAL_MODEL].values
    thr, macro_f1 = tune_threshold(cal_pred["y_true"], proba_cal)

    proba_test = test_pred[FINAL_MODEL].values
    metrics = evaluate_at_threshold(test_pred["y_true"], proba_test, thr)
    auc_point, auc_lo, auc_hi = bootstrap_ci(test_pred["y_true"], proba_test,
                                              n_resamples=N_BOOTSTRAP, seed=SEED)

    print(f"{dataset_tag}: threshold={thr:.2f}  Acc={metrics['accuracy']:.4f}  "
          f"MacroF1={metrics['macro_f1']:.4f}  AUC={metrics['auc_roc']:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

    return {
        "dataset": dataset_tag, "proposed_model": FINAL_MODEL, "threshold": round(thr, 2),
        "accuracy": round(metrics["accuracy"], 4), "macro_f1": round(metrics["macro_f1"], 4),
        "auc_roc": round(metrics["auc_roc"], 4), "auc_ci_lower": round(auc_lo, 4),
        "auc_ci_upper": round(auc_hi, 4),
    }

final_results = []
for tag, ddir in [("pakistan", os.path.join(PROCESSED_DIR, "pakistan")),
                   ("arab5_depression", os.path.join(PROCESSED_DIR, "arab5_depression")),
                   ("arab5_anxiety", os.path.join(PROCESSED_DIR, "arab5_anxiety")),
                   ("uganda", os.path.join(PROCESSED_DIR, "uganda"))]:
    final_results.append(finalize_proposed_model(tag, ddir))

final_results_df = pd.DataFrame(final_results)
print(final_results_df)
final_results_df.to_csv(os.path.join(TABLES_DIR, "proposed_model_final_results.csv"), index=False)

pakistan: threshold=0.51  Acc=0.5486  MacroF1=0.5329  AUC=0.5571 [0.5353, 0.5792]
arab5_depression: threshold=0.55  Acc=0.6379  MacroF1=0.6314  AUC=0.6867 [0.6252, 0.7468]
arab5_anxiety: threshold=0.45  Acc=0.7958  MacroF1=0.7228  AUC=0.7815 [0.6958, 0.8631]
uganda: threshold=0.55  Acc=0.6364  MacroF1=0.6314  AUC=0.6965 [0.6770, 0.7162]
            dataset proposed_model  threshold  accuracy  macro_f1  auc_roc  \
0          pakistan             rf       0.51    0.5486    0.5329   0.5571   
1  arab5_depression             rf       0.55    0.6379    0.6314   0.6867   
2     arab5_anxiety             rf       0.45    0.7958    0.7228   0.7815   
3            uganda             rf       0.55    0.6364    0.6314   0.6965   

   auc_ci_lower  auc_ci_upper  
0        0.5353        0.5792  
1        0.6252        0.7468  
2        0.6958        0.8631  
3        0.6770        0.7162  


In [10]:
from sklearn.metrics import roc_auc_score

justification_rows = []
for tag, ddir in [("pakistan", os.path.join(PROCESSED_DIR, "pakistan")),
                   ("arab5_depression", os.path.join(PROCESSED_DIR, "arab5_depression")),
                   ("arab5_anxiety", os.path.join(PROCESSED_DIR, "arab5_anxiety")),
                   ("uganda", os.path.join(PROCESSED_DIR, "uganda"))]:
    oof = pd.read_csv(os.path.join(ddir, "oof_predictions_train.csv"))
    aucs = {name: roc_auc_score(oof["y_true"], oof[name]) for name in MODEL_NAMES}
    ranked = sorted(aucs.items(), key=lambda x: -x[1])
    rf_rank = [i for i, (name, _) in enumerate(ranked, 1) if name == "rf"][0]
    justification_rows.append({
        "dataset": tag, "rf_oof_auc": round(aucs["rf"], 4),
        "best_oof_model": ranked[0][0], "best_oof_auc": round(ranked[0][1], 4),
        "rf_rank_among_7": rf_rank,
        "gap_to_best": round(ranked[0][1] - aucs["rf"], 4),
    })

justification_df = pd.DataFrame(justification_rows)
print(justification_df)
justification_df.to_csv(os.path.join(TABLES_DIR, "rf_selection_justification.csv"), index=False)

            dataset  rf_oof_auc best_oof_model  best_oof_auc  rf_rank_among_7  \
0          pakistan      0.5382            svm        0.5418                3   
1  arab5_depression      0.6868             rf        0.6868                1   
2     arab5_anxiety      0.7718             rf        0.7718                1   
3            uganda      0.6913            xgb        0.6916                2   

   gap_to_best  
0       0.0036  
1       0.0000  
2       0.0000  
3       0.0002  
